# QuEst dye quenching and FRET demo (headless)

This notebook is a port of the legacy `mfm`-based *Dye-quenching* example to the current QuEst backend.

It reproduces the same conceptual steps as the original notebook:

- **Structure-based accessible volume (AV)** around a labeled residue on hGBP1.
- **Diffusion-based dye quenching** by protein residues within a critical distance.
- **Photon-based donor decay simulation** and quantum yield estimate.
- **Optional FRET channel** using an acceptor dye on the same structure and a Förster radius.

All calculations are performed via the GUI-free `quest.lib.tools.dye_diffusion.dye_diffusion.DonorDecay` API so the notebook can run headless.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as p

# Resolve repository root when running from repo root or this notebook folder
try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path().resolve()
    if not (ROOT / "modules" / "quest").exists():
        ROOT = Path().resolve().parents[2]

QUEST_ROOT = ROOT / "modules" / "quest"
if str(QUEST_ROOT) not in sys.path:
    sys.path.insert(0, str(QUEST_ROOT))

from quest.lib.tools.dye_diffusion.dye_diffusion import DonorDecay

PDB_HGBP1_CLOSED = ROOT / "test" / "data" / "atomic_coordinates" / "pdb_files" / "hGBP1_closed.pdb"


# --- Donor-only dye quenching -------------------------------------------------

# Parameters chosen to mirror the original legacy example as closely as possible.
# - hGBP1 closed structure
# - Attachment at residue 344 CB
# - Surface sticking with slow_radius / slow_fact
# - All quenching residues treated as potential quenchers within critical_distance

donor = DonorDecay(
    tau0=4.0,
    kQ=0.5,
    nph=24_000_000,  # photons for "photon"-style decay
    verbose=False,
    auto_update=False,
    pdb=None,
    attachment_residue=344,
    attachment_atom="CB",
    attachment_chain=None,  # no explicit chain in hGBP1_closed.pdb
    dg=0.5,
    sticky_mode="surface",
    save_avs=False,
    D=23.0,
    slow_fact=0.6,
    critical_distance=5.1,
    output_file="dye_quenching_demo",
    av_parameter={"linker_length": 20.0, "linker_width": 0.5, "radius1": 2.0},
    quencher={"TRP": ["CB"], "TYR": ["CB"], "HIS": ["CB"]},
    t_max=500.0,
    t_step=0.032,
    slow_radius=8.5,
    all_quencher_atoms=True,
)

donor.structure = str(PDB_HGBP1_CLOSED)

# One full update performs: AV -> slow AV -> diffusion -> photon trace
success = donor.update_all(verbose=False)
if not success:
    raise RuntimeError("Diffusion trajectory simulation failed for donor dye.")

# "Curve" vs "photon" styles: reuse the same diffusion but vary photon statistics.
# First, a moderate-photon curve-style decay.

donor.n_photons = 1_000_000
# Reset photon trace so it will be recomputed
try:
    donor._photon_trace = None  # type: ignore[attr-defined]
except Exception:
    pass
x_curve, y_curve = donor.get_histogram(nbins=4096, range=(0, 50))
y_curve = y_curve.astype(float)
y_curve_norm = y_curve / (y_curve.max() if y_curve.max() > 0 else 1.0)

# Now a high-statistics "photon"-style decay.

donor.n_photons = 24_000_000
try:
    donor._photon_trace = None  # type: ignore[attr-defined]
except Exception:
    pass
x_photon, y_photon = donor.get_histogram(nbins=4096, range=(0, 50))
y_photon = y_photon.astype(float)
y_photon_norm = y_photon / (y_photon.max() if y_photon.max() > 0 else 1.0)

print(f"Quantum yield (donor): {donor.quantum_yield:.3f}")
print(f"Collisions fraction: {donor.collisions:.3f}")

# Plot donor curve vs photon decays on a semilogarithmic scale
p.figure(figsize=(6, 4))
p.semilogy(x_curve, y_curve_norm + 1e-4, label="curve-style")
p.semilogy(x_photon, y_photon_norm + 1e-4, label="photon-style")
p.xlabel("time (ns)")
p.ylabel("normalized counts")
p.legend()
p.tight_layout()
p.show()

# Time axis for the underlying diffusion trajectory (analogous to diffusion_donor.time)
traj = donor.diffusion._traj  # type: ignore[attr-defined]
time_axis = np.arange(traj.shape[0]) * donor.diffusion.t_step


# --- FRET-decay (optional) ----------------------------------------------------

# Set up a donor/acceptor pair on the same structure, approximating the
# legacy example (donor at residue 254, acceptor at residue 577, both CB).

fret = DonorDecay(
    tau0=4.0,
    kQ=0.5,
    nph=1_000_000,
    verbose=False,
    auto_update=False,
    pdb=None,
    attachment_residue=254,
    attachment_atom="CB",
    attachment_chain=None,
    dg=0.5,
    sticky_mode="surface",
    save_avs=False,
    D=23.0,
    slow_fact=0.6,
    critical_distance=5.1,
    output_file="dye_quenching_fret_demo",
    av_parameter={"linker_length": 20.0, "linker_width": 0.5, "radius1": 2.0},
    quencher={"TRP": ["CB"], "TYR": ["CB"], "HIS": ["CB"]},
    t_max=500.0,
    t_step=0.032,
    slow_radius=8.5,
    all_quencher_atoms=True,
)

fret.structure = str(PDB_HGBP1_CLOSED)

# Enable FRET: acceptor AV, Förster radius, and acceptor attachment site.
fret.fret_enabled = True
fret.fret_R0 = 52.0
fret.acceptor_chain = None
fret.acceptor_residue = 577
fret.acceptor_atom = "CB"

success_fret = fret.update_all(verbose=False)
if not success_fret:
    raise RuntimeError("Diffusion trajectory simulation failed for FRET pair.")

x_fret, y_fret = fret.get_histogram_fret(nbins=4096, range=(0, 50))
if x_fret is not None and y_fret is not None:
    y_fret = y_fret.astype(float)
    y_fret_norm = y_fret / (y_fret.max() if y_fret.max() > 0 else 1.0)

    print(f"Quantum yield (FRET channel): {fret.quantum_yield_fret:.3f}")
    print(f"FRET efficiency (Donor->Acceptor): {fret.fret_efficiency:.3f}")

    p.figure(figsize=(6, 4))
    p.semilogy(x_curve, y_curve_norm + 1e-4, label="donor (no FRET)")
    p.semilogy(x_fret, y_fret_norm + 1e-4, label="FRET channel")
    p.xlabel("time (ns)")
    p.ylabel("normalized counts")
    p.legend()
    p.tight_layout()
    p.show()
else:
    print("FRET decay could not be computed (no acceptor AV or invalid geometry).")
